# Compass Dashboard

In [1]:
import pandas as pd

In [3]:
shift_data = {
    'month':['march','march','march'],
    'day':['Sunday','Monday','Tuesday'],
    'start':[1445,1445,1445],
    'end':[1835,1835,1835]
}

shifts = pd.DataFrame(shift_data)
shifts

,month,day,start,end
0,march,Sunday,1445,1835
1,march,Monday,1445,1835
2,march,Tuesday,1445,1835


In [4]:
fare_data = {
    '1zone': 2.70,
    '2zone': 4.00,
    '1zonemonth': 111.60,
    '2zonemonth': 149.25,
    '2zoneaddfare': 1.50,
    'yvraddfare' : 5.00
}
fare_data

{'1zone': 2.7,
 '2zone': 4.0,
 '1zonemonth': 111.6,
 '2zonemonth': 149.25,
 '2zoneaddfare': 1.5,
 'yvraddfare': 5.0}

In [5]:
def ShiftsToTrips(df:pd.DataFrame):
    "Convert a shifts schedule to the implied commute schedule"
    trips = df.copy()
    trips['start'] = df['start'] - 100
    trips.rename(columns={'start':'home','end':'YVR'}, inplace=True)
    trips = trips.melt(id_vars=['month','day'],
                        value_vars=['home','YVR'],
                        var_name='origin',
                        value_name='time')
    return trips

In [8]:
trips = ShiftsToTrips(shifts)
trips

,month,day,origin,time
0,march,Sunday,home,1345
1,march,Monday,home,1345
2,march,Tuesday,home,1345
3,march,Sunday,YVR,1835
4,march,Monday,YVR,1835
5,march,Tuesday,YVR,1835


In [9]:
def GetOneZoneTrips(df:pd.DataFrame):
    "Find all trips that occur on a weekend or after 6:30pm"
    one_zone = df.loc[(df['day'].isin(['Saturday','Sunday'])) | (df['time']>1800)]

    return one_zone

GetOneZoneTrips(trips)

,month,day,origin,time
0,march,Sunday,home,1345
3,march,Sunday,YVR,1835
4,march,Monday,YVR,1835
5,march,Tuesday,YVR,1835


In [10]:
def GetTwoZoneTrips(df:pd.DataFrame):
    "find all trips that occur on a weekday before 6:30pm"
    two_zone = df.loc[(df['day'].isin(['Monday','Tuesday','Wednesday','Thursday','Friday']) & (df['time']<1800))]

    return two_zone

GetTwoZoneTrips(trips)

,month,day,origin,time
1,march,Monday,home,1345
2,march,Tuesday,home,1345


In [11]:
def GetAddFareTrips(df:pd.DataFrame):
    "find all trips that will incur the YVR add fare"
    add_fare = df.loc[df['origin'] == 'YVR']

    return add_fare

GetAddFareTrips(trips)

,month,day,origin,time
3,march,Sunday,YVR,1835
4,march,Monday,YVR,1835
5,march,Tuesday,YVR,1835


In [12]:
def TotalCost(shifts:pd.DataFrame, fares:dict, method:str):
    "find the total cost of commuting based on the given method"

    # convert shifts to trips
    trips = ShiftsToTrips(shifts)

    # count one zone trips
    OneZone = GetOneZoneTrips(trips).shape[0]

    # count two zone trips
    TwoZone = GetTwoZoneTrips(trips).shape[0]

    # count add fare trips
    AddFare = GetAddFareTrips(trips).shape[0]

    if method == 'StoredValue':
        TotalCost = OneZone*fares['1zone'] + TwoZone*fares['2zone'] + AddFare*fares['yvraddfare']
        return TotalCost
    if method == 'OneZoneMonthly':
        TotalCost = fares['1zonemonth'] + TwoZone*fares['2zoneaddfare']
        return TotalCost
    if method == 'TwoZoneMonthly':
        TotalCost = fares['2zonemonth']
        return TotalCost

    return None

TotalCost(shifts, fare_data, 'StoredValue')

33.8

In [13]:
CommuteCost = {
    'Stored Value' : TotalCost(shifts, fare_data, 'StoredValue'),
    '1 Zone Month Pass' : TotalCost(shifts, fare_data, 'OneZoneMonthly'),
    '2 Zone Month Pass' : TotalCost(shifts, fare_data, 'TwoZoneMonthly')
}

pd.DataFrame(CommuteCost, index=range(1))

,Stored Value,1 Zone Month Pass,2 Zone Month Pass
0,33.8,114.6,149.25


## Preprocessing


In [5]:
import pdfplumber

with pdfplumber.open("../data/Monthly Schedule _ MAR.pdf") as pdf:
    page = pdf.pages[0]
    table = page.extract_table()

print(table)

[['MAR 2026', 'Mon\n02-Mar', 'Tue\n03-Mar', 'Wed\n04-Mar', 'Thu\n05-Mar', 'Fri\n06-Mar', 'Sat\n07-Mar', 'Sun\n08-Mar'], ['Ratna\nSalim', '', '1445-1845\nAGT1', '', '', '1445-1845\nAGT1', '', ''], ['Alex\nYe', '', '', '', '', '', '', ''], ['Ann\nKim', '', '1445-1845\nAGT1', '1445-1845\nAGT1', '', '1415-1815\nTKT', '', '1545-1945\nAGT1'], ['Saeko\nSumi', '', '1215-1915 FC', '1215-1915 FC', '1215-1915 FC', '1215-1915 FC', '1215-1915 FC', ''], ['Joey\nLu', '1115-1915\nSUP', '1115-1915\nSUP', '', '1115-1915\nSUP', '', '1115-1915\nSUP', '1215-2015\nOPS'], ['Cindy\nXiao', '', '', '1145-1845\nARR+DEP', '', '1445-1845\nAGT1', '', '1245-1945\nARR+DEP'], ['Satomi\nHarward', '1145-1845\nARR+DEP', '1145-1845\nARR+DEP', '', '', '1145-1845\nARR+DEP', '1145-1845\nARR+DEP', '1245-1945\nARR+DEP'], ['Kohei\nTodoroki', '', '1415-1815\nTKT', '', '1415-1815\nTKT', '', '', '1515-1915\nTKT'], ['Rich\nHung', '', '', '', '1445-1845\nAGT1', '', '1445-1845\nAGT1', ''], ['Cecillia\nZhu', '', '1145-1845 WT', '1145-

In [6]:
pd.DataFrame(table)

,0,1,2,3,4,5,6,7
0,MAR 2026,Mon\n02-Mar,Tue\n03-Mar,Wed\n04-Mar,Thu\n05-Mar,Fri\n06-Mar,Sat\n07-Mar,Sun\n08-Mar
1,Ratna\nSalim,,1445-1845\nAGT1,,,1445-1845\nAGT1,,
2,Alex\nYe,,,,,,,
3,Ann\nKim,,1445-1845\nAGT1,1445-1845\nAGT1,,1415-1815\nTKT,,1545-1945\nAGT1
4,Saeko\nSumi,,1215-1915 FC,1215-1915 FC,1215-1915 FC,1215-1915 FC,1215-1915 FC,
5,Joey\nLu,1115-1915\nSUP,1115-1915\nSUP,,1115-1915\nSUP,,1115-1915\nSUP,1215-2015\nOPS
6,Cindy\nXiao,,,1145-1845\nARR+DEP,,1445-1845\nAGT1,,1245-1945\nARR+DEP
7,Satomi\nHarward,1145-1845\nARR+DEP,1145-1845\nARR+DEP,,,1145-1845\nARR+DEP,1145-1845\nARR+DEP,1245-1945\nARR+DEP
8,Kohei\nTodoroki,,1415-1815\nTKT,,1415-1815\nTKT,,,1515-1915\nTKT
9,Rich\nHung,,,,1445-1845\nAGT1,,1445-1845\nAGT1,


In [8]:
with pdfplumber.open("../data/Monthly Schedule _ MAR.pdf") as pdf:
    page = pdf.pages[0]
    text = page.extract_text()

print(text)

Mon Tue Wed Thu Fri Sat Sun
MAR 2026
02-Mar 03-Mar 04-Mar 05-Mar 06-Mar 07-Mar 08-Mar
Ratna 1445-1845 1445-1845
Salim AGT1 AGT1
Alex
Ye
Ann 1445-1845 1445-1845 1415-1815 1545-1945
Kim AGT1 AGT1 TKT AGT1
Saeko
1215-1915 FC 1215-1915 FC 1215-1915 FC 1215-1915 FC 1215-1915 FC
Sumi
Joey 1115-1915 1115-1915 1115-1915 1115-1915 1215-2015
Lu SUP SUP SUP SUP OPS
Cindy 1145-1845 1445-1845 1245-1945
Xiao ARR+DEP AGT1 ARR+DEP
Satomi 1145-1845 1145-1845 1145-1845 1145-1845 1245-1945
Harward ARR+DEP ARR+DEP ARR+DEP ARR+DEP ARR+DEP
Kohei 1415-1815 1415-1815 1515-1915
Todoroki TKT TKT TKT
Rich 1445-1845 1445-1845
Hung AGT1 AGT1
Cecillia
1145-1845 WT 1145-1845 WT 1145-1845 WT 1145-1845 WT 1245-1945 WT
Zhu
Donia 1445-1845 1445-1845 1445-1845
Forbes AGT1 AGT1 AGT1
Kevin 1145-1845 1145-1845
Lee ARR+DEP ARR+DEP
Crystal 1145-1845 1145-1845 1145-1845
1215-1915 FC 1315-2015 FC
Deng ARR+DEP ARR+DEP ARR+DEP
Eunsol 1445-1845 1445-1845 1445-1845 1445-1845
Choi AGT1 AGT1 AGT1 AGT1
Angel 1415-1815 1145-1815 1145-1

So in fact with this specific document it is quite easy to use extract_table and turn it directly into a data frame. However, with PDF documents this is not always the case, and a more robust solution that is good to practice is to extract words with their positions and reconstruct the table using those relative postions.

In [9]:
with pdfplumber.open("../data/Monthly Schedule _ MAR.pdf") as pdf:
    page = pdf.pages[0]
    words = page.extract_words()

for w in words[198:210]:
    print(w)

{'text': 'Ian', 'x0': 20.4, 'x1': 34.012319999999995, 'top': 643.17328, 'doctop': 643.17328, 'bottom': 654.2132799999999, 'upright': True, 'height': 11.039999999999964, 'width': 13.612319999999997, 'direction': 'ltr'}
{'text': '1445-1845', 'x0': 88.08, 'x1': 138.01312, 'top': 642.09328, 'doctop': 642.09328, 'bottom': 653.13328, 'upright': True, 'height': 11.039999999999964, 'width': 49.93311999999999, 'direction': 'ltr'}
{'text': '1445-1845', 'x0': 232.37, 'x1': 282.30312, 'top': 642.09328, 'doctop': 642.09328, 'bottom': 653.13328, 'upright': True, 'height': 11.039999999999964, 'width': 49.933119999999974, 'direction': 'ltr'}
{'text': '1445-1845', 'x0': 304.49, 'x1': 354.42312, 'top': 642.09328, 'doctop': 642.09328, 'bottom': 653.13328, 'upright': True, 'height': 11.039999999999964, 'width': 49.933119999999974, 'direction': 'ltr'}
{'text': '1445-1845', 'x0': 448.78, 'x1': 498.71311999999995, 'top': 642.09328, 'doctop': 642.09328, 'bottom': 653.13328, 'upright': True, 'height': 11.03999

In [10]:
def extract_words(pdf_path):
    with pdfplumber.open(pdf_path) as pdf:
        page = pdf.pages[0]
        words = page.extract_words()

    return words

In [11]:
import re

DATE_PATTERN = re.compile(r"\d{2}-[A-Za-z]{3}")

def get_date_columns(words):
    date_columns = []

    for w in words:
        if DATE_PATTERN.match(w["text"]):
            date_columns.append({
                "date": w["text"],
                "x": w["x0"]
            })

    date_columns = sorted(date_columns, key=lambda d: d["x"])

    return date_columns

In [12]:
date_columns = get_date_columns(words)
date_columns

[{'date': '02-Mar', 'x': 96.96},
 {'date': '03-Mar', 'x': 169.1},
 {'date': '04-Mar', 'x': 241.25},
 {'date': '05-Mar', 'x': 313.37},
 {'date': '06-Mar', 'x': 385.51},
 {'date': '07-Mar', 'x': 457.66},
 {'date': '08-Mar', 'x': 529.8}]

In [13]:
def find_employee_rows(words, first="Ian", last= "Maccarthy", y_tolerance=10):

    rows = {}

    for i,w in enumerate(words):
        if w["text"] == first:
            next_line = words[i:i+tolerance]
            
            for v in next_line:               
                if v['text'] == last:                    
                    rows['shift time row'] = w['top']
                    rows['shift type row'] = v['top']
                    return rows
            
    raise ValueError("Employee not found, or names separated beyond tolerance")

So that would work but it assumes that the first and last names of the employee will be more or less consecutive in the list of parsed words. This is true in the present case, but not can't be relied on in general. In the following version we use only geometry and search the entire word list for the first and last names.

In [22]:
def find_employee_rows(words, first="Ian", last="Maccarthy", y_tol=20, x_tol=20):
    firstName = None
    lastName = None

    for w in words:
        if w['text'] == first:
            firstName = w
            break

    if not firstName:
        raise ValueError("First name not found")
    
    for w in words:
        if (
            w['text'] == last
            and abs(w['x0'] - firstName['x0']) < x_tol
            and w['top'] > firstName['top']
            and abs(w['top'] - firstName['top']) < y_tol
        ):
            lastName = w
            break
    if not lastName:
        raise ValueError("last name not found")
    
    return{
        "shiftTimeRow": firstName['top'],
        "shiftTypeRow": lastName['top']
    }

In [23]:
Ian_rows = find_employee_rows(words, "Ian", "Maccarthy", 20, 20)
Ian_rows

{'shiftTimeRow': 643.17328, 'shiftTypeRow': 657.09328}

In [19]:
657-643


14

In [20]:
SHIFT_PATTERN = re.compile(r"\d{4}-\d{4}")

def get_employee_shifts(words, employee_y, tolerance=20):
    shifts = []

    for w in words:
        if SHIFT_PATTERN.match(w['text']):

            if abs(w['top'] - employee_y) < tolerance:
                shifts.append(w)

    return shifts

In [24]:
Ian_shifts = get_employee_shifts(words, Ian_rows['shiftTimeRow'])
for x in Ian_shifts:
    print(x['text'],x['x0'],x['top'])

1445-1845 88.08 642.09328
1445-1845 232.37 642.09328
1445-1845 304.49 642.09328
1445-1845 448.78 642.09328
1545-1945 520.9 642.09328


Now we have the y coordinates for shift type and time (sometimes they will actually be the same), and we can match shift types to their times simply by geometric proximity rather than a new regex search

In [29]:
def match_shift_types(shifts, words, shift_type_y, y_tol=20, x_tol=20):
    results = []

    for shift in shifts:
        x = shift['x0']

        for w in words:
            if (
                abs(w['top'] - shift_type_y) < y_tol and 
                abs(w['x0'] - x) < x_tol and
                not SHIFT_PATTERN.match(w['text'])
            ):
                results.append({
                    'x': x,
                    'time': shift['text'],
                    'type': w['text']
                })
                break

    return results

In [32]:
Ian_shifts_with_type  = match_shift_types(Ian_shifts, words, Ian_rows['shiftTypeRow'], 20, 20)

Ian_shifts_with_type

[{'x': 88.08, 'time': '1445-1845', 'type': 'AGT1'},
 {'x': 232.37, 'time': '1445-1845', 'type': 'AGT1'},
 {'x': 304.49, 'time': '1445-1845', 'type': 'AGT1'},
 {'x': 448.78, 'time': '1445-1845', 'type': 'AGT1'},
 {'x': 520.9, 'time': '1545-1945', 'type': 'AGT1'}]

In [42]:
def assign_dates(shifts, date_columns):

    results = []

    for shift in shifts:
        x = shift["x"]

        closest = min(
            date_columns,
            key=lambda d: abs(d['x']-x)
        )

        results.append({
            "date": closest["date"],
            "time": shift["time"],
            "type": shift["type"]
        })

    return results

In [43]:
results = assign_dates(Ian_shifts_with_type, date_columns)
results

[{'date': '02-Mar', 'time': '1445-1845', 'type': 'AGT1'},
 {'date': '04-Mar', 'time': '1445-1845', 'type': 'AGT1'},
 {'date': '05-Mar', 'time': '1445-1845', 'type': 'AGT1'},
 {'date': '07-Mar', 'time': '1445-1845', 'type': 'AGT1'},
 {'date': '08-Mar', 'time': '1545-1945', 'type': 'AGT1'}]

In [57]:
def build_dataframe(results):

    data = []

    for r in results:
        start, end = r["time"].split("-")

        data.append({
            "date":r["date"],
            "type": r['type'],
            "start": start,
            "end": end
        })

    return pd.DataFrame(data)

In [58]:
df = build_dataframe(results)
df

,date,type,start,end
0,02-Mar,AGT1,1445,1845
1,04-Mar,AGT1,1445,1845
2,05-Mar,AGT1,1445,1845
3,07-Mar,AGT1,1445,1845
4,08-Mar,AGT1,1545,1945
5,02-Mar,AGT1,1545,1945
6,04-Mar,AGT1,1545,1945
7,05-Mar,AGT1,1245,1945
8,07-Mar,AGT1,1545,1945
9,08-Mar,AGT1,1545,1945


so far we have just parsed one page of the original schedule document. The next step is to parse the entire monthly schedule

In [59]:
def parse_pdf(pdf_path):
    all_results = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            words = page.extract_words()

            date_columns = get_date_columns(words)
            employee_rows = find_employee_rows(words)
            shifts = get_employee_shifts(words, employee_rows['shiftTimeRow'])
            shifts_w_type = match_shift_types(
                shifts, words, employee_rows['shiftTypeRow']
                )
            results = assign_dates(shifts_w_type, date_columns)

            all_results.extend(results)

    return all_results

In [68]:
def add_datetime(df, year=2026):
    df["date"] = pd.to_datetime(
        df["date"] + f"-{year}"
    )
    return df

which is fine except that hardcoding the year will fail once it rolls over in january. the following version of add_datetime() will automatically increment the year when this happens

In [74]:
def add_datetime(df, start_year = 2026):
    dates = df['date'].to_list()

    parsedDates = []
    currentYear = start_year
    prevMonth = None

    for d in dates:
        dt = pd.to_datetime(d, format="%d-%b")
        month = dt.month

        if prevMonth == 12 and month == 1:
            currentYear += 1

        parsedDates.append(dt.replace(year=currentYear))
        prevMonth = month

    df["date"] = parsedDates
    return df


In [70]:
def add_weekday_info(df):
    df['weekday'] = df["date"].dt.day_name()
    df["is_weekend"] = df["date"].dt.weekday >=5
    return df

In [72]:
def add_month(df):
    df['month'] = df['date'].dt.month_name()
    return df

In [76]:
data = parse_pdf('../data/Monthly Schedule _ MAR.pdf')
df = build_dataframe(data)
df = add_datetime(df)
df = add_weekday_info(df)
df = add_month(df)
df = df.sort_values('date').reset_index(drop=True)
df

,date,type,start,end,weekday,is_weekend,month
0,2026-03-02,AGT1,1445,1845,Monday,False,March
1,2026-03-04,AGT1,1445,1845,Wednesday,False,March
2,2026-03-05,AGT1,1445,1845,Thursday,False,March
3,2026-03-07,AGT1,1445,1845,Saturday,True,March
4,2026-03-08,AGT1,1545,1945,Sunday,True,March
5,2026-03-09,AGT1,1545,1945,Monday,False,March
6,2026-03-11,AGT1,1545,1945,Wednesday,False,March
7,2026-03-12,ARR+DEP,1245,1945,Thursday,False,March
8,2026-03-14,AGT1,1545,1945,Saturday,True,March
9,2026-03-15,AGT1,1545,1945,Sunday,True,March
